### Análisis de sentimiento de los títulos de noticias

Más información sobre sentiment analysis, acá: https://www.datacamp.com/tutorial/text-analytics-beginners-nltk

In [ ]:
#Si aún no instalaron estas librerías, activar estas líneas de código -quitar #- para instalarlas
#!pip install string
#!pip install pandas
#!pip install nltk
#!pip install stop_words
#!pip install spacy
#!python -m spacy download es_core_news_sm
#!pip uninstall vaderSentiment
#!pip install vader-multi

In [ ]:
#Importamos los paquetes a utilizar
import string
import pandas as pd
import nltk #para procesamiento del lenguaje natural
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from stop_words import get_stop_words
from nltk.stem import WordNetLemmatizer
import spacy #para preprocesamiento en español
from nltk.sentiment.vader import SentimentIntensityAnalyzer

#ntlk requiere descargar algunos datos adicionales
nltk.download('all')

#Para trabajar en inglés usar:
from textblob import TextBlob

In [ ]:
from stop_words import get_stop_words

In [ ]:
#Cargamos la base de datos que creamos en la tutorial
titulos = pd.read_excel('titulos.xlsx')
titulos

Vamos a limpiar los títulos. Como parte del preprocesamiento de la información tenemos que realizar los siguientes pasos:
1. Tokenization: Implica dividir el texto en palabras (o tokens)
2. Eliminar stop words: quitar palabras comunes e irrelevantes que no tienen mucho "sentimiento". Esto permite mejorar la precisión del análisis de sentimiento
3. Lemmatization: reducir las palabras a sus raíces (por ejemplo, pasar de "leyendo" a "leer" al eliminar los sufijos).

In [ ]:
#Veamos la lista de signos de puntuación
print(string.punctuation)
#Como estamos trabajando en español, es conveniente agregar algunos símbolos más
# a los signos de puntuación
string.punctuation = string.punctuation + '¿¡“”'
print(string.punctuation)

#Cargamos palabras vacías en español
stop_words = get_stop_words('spanish')
print(stop_words)

In [ ]:
def limpiar_titulos_nltk(titulo):
    '''
    Esta función limpia el texto del título.
    Convierte texto en tokens, elimina stop words, y transforma palabras en su forma raíz
    para dejar en el texto solo las palabras con mayor contenido.
    Input:
        título (str): Texto del título original
    Output:
        título (str): Texto del título limpio
    '''

    #Separamos los títulos en tokens (obtenemos una lista con palabras)
    word_tokens = word_tokenize(titulo.lower())

    #Eliminamos palabras vacías (stop words) de los títulos
    # Loop por las condiciones
    filtered_tokens = []
    for w in word_tokens:
        # Verificamos tokens contra stop words y puntuación
        if w not in stop_words and w not in string.punctuation:
            filtered_tokens.append(w) #La agregamos a la lista

    #Lemmatization
    lemmatizer = WordNetLemmatizer()

    lemmatized_tokens = []
    for w in filtered_tokens:
        lemmatizer.lemmatize(w)
        lemmatized_tokens.append(w)

    #Volvemos a armar la oración (concatenamos las palabras separándolas con un espacio)
    return ' '.join(lemmatized_tokens)

In [ ]:
#Cargamos el modelo para el español y las stop words según spacy
nlp = spacy.load('es_core_news_sm')
stopwords_spacy = spacy.lang.es.stop_words.STOP_WORDS

def limpiar_titulos_spacy(titulo):
    '''
    Esta función limpia el texto del título (usando funcionalidades de la librería spacy).
    Convierte texto en tokens, elimina stop words, y transforma palabras en su forma raíz
    para dejar en el texto solo las palabras con mayor contenido.
    Input:
        título (str): Texto del título original
    Output:
        título (str): Texto del título limpio
    '''

    #Procesamos el texto con spaCy
    doc = nlp(titulo.lower())
    #print(doc)

    filtered_tokens = []
    lemmas = []

    #Pasamos a tokens y eliminamos puntación y stopwords
    for w in doc:
        if w.text not in stopwords_spacy and not w.is_punct:
            filtered_tokens.append(w.text)
    filtered_doc = ' '.join(filtered_tokens)
    print(filtered_doc)

    #Obtenemos las formas lematizadas de las palabras
    doc2 = nlp(filtered_doc)
    for w_f in doc2:
        lemmas.append(w_f.lemma_)

    #Volvemos a armar la oración (concatenamos las palabras separándolas con un espacio)
    return ' '.join(lemmas)

In [ ]:
#Este es un título sucio:
titulos[0]

In [ ]:
#Este es un título limpio:
limpiar_titulos_nltk(titulos[0]['titular'])

In [ ]:
#Este es un título limpio:
limpiar_titulos_spacy(titulos[0]['titular'])

#### Ahora vamos a usar sentiment analysis para ver qué tan positivos son los títulos

Vamos a usar la bilioteca NLTK (Natural Language Toolkit) para clasificar los títulos en positivos o negativos. NLTK es una biblioteca de Python muy utilizada en procesamiento de lenguaje natural (NLP). VADER (Valence Aware Dictionary and Sentiment Reasoner) es un módulo específico de NLTK que se utiliza para el análisis de sentimientos.

VADER es una herramienta especialmente diseñada para el análisis de sentimientos en textos. A diferencia de algunos enfoques más generales que utilizan modelos de aprendizaje automático, VADER se basa en un conjunto de reglas y un diccionario que asigna puntuaciones de polaridad a palabras y expresiones. Además, tiene en cuenta factores como las mayúsculas, los signos de puntuación y los emoticonos para evaluar la intensidad del sentimiento.

Las puntuaciones de VADER incluyen la polaridad (positiva, negativa o neutra) y una medida de la intensidad del sentimiento. Es especialmente útil para textos informales o con lenguaje coloquial, como se encuentra comúnmente en redes sociales.

Vamos a usar un módulo de VADER "multi", que resuelve la misma tarea pero con otro modelo. Ver: https://github.com/brunneis/vader-multi

In [ ]:
#También pueden usar VADER "original" (Ver: https://www.nltk.org/_modules/nltk/sentiment/vader.html).
# from nltk.sentiment.vader import SentimentIntensityAnalyzer

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer as SentimentIntensityAnalyzer

#Inicializamos el analizador de sentimientos VADER
sia = SentimentIntensityAnalyzer()

#Primero veamos un ejemplo
texto_ej_pos = "Me encanta este curso, es genial"
texto_ej_neg = "Odio este curso, es terrible"
texto_ej_neu = "Este curso me da igual"

print(texto_ej_pos, sia.polarity_scores(texto_ej_pos))
print(texto_ej_neg, sia.polarity_scores(texto_ej_neg))
print(texto_ej_neu, sia.polarity_scores(texto_ej_neu))
#Si la variable compound es positiva, el texto es positivo; si es negativa, el texto es negativo
# Y si se encuentra en el rango del 0 es un mensaje neutro

Ahora crearemos funciones que, además de dar un valor, clasifiquen en positivo, negativo o neutro

In [ ]:
def analizar_sentiment(text):
    #Obtenemos la polaridad del sentimiento
    sia = SentimentIntensityAnalyzer()
    sentiment_score = sia.polarity_scores(text)

    #Clasificamos el sentimiento como positivo, negativo o neutro
    compound_score = sentiment_score['compound']
    if compound_score >= 0.05:
        sentiment = "Positivo"
    elif compound_score <= -0.05:
        sentiment = "Negativo"
    else:
        sentiment = "Neutro"

    return compound_score, sentiment

In [ ]:
#Ejemplos de uso
compound_score_pos, sentiment_pos = analizar_sentiment(texto_ej_pos)
print(f"Texto: {texto_ej_pos}")
print(f"Puntuación de sentimiento compuesta: {compound_score_pos}")
print(f"Sentimiento: {sentiment_pos}")

compound_score_neg, sentiment_neg = analizar_sentiment(texto_ej_neg)
print(f"\nTexto: {texto_ej_neg}")
print(f"Puntuación de sentimiento compuesta: {compound_score_neg}")
print(f"Sentimiento: {sentiment_neg}")

In [ ]:
#Ahora un ejemplo con un título limpio
print(titulos[0]['titular'],
      "\n",
      analizar_sentiment(limpiar_titulos_spacy(titulos[0]['titular'])))


In [ ]:
titulos[0]

In [ ]:
#Aplicamos la función para limpiar títulos para tener un columna con títulos limpios
titulos_df['titular_limpio'] = titulos_df['titular'].apply(limpiar_titulos_nltk)
#Vemos el sentiment
titulos_df['sentiment'] = titulos_df['titular_limpio'].apply(analizar_sentiment)

In [ ]:
titulos_df

In [ ]:
#Separamos en dos columnas
titulos_df[['value', 'emotion']] = pd.DataFrame(titulos_df['sentiment'].tolist(), index=titulos_df.index)
titulos_df_final = titulos_df.drop('sentiment', axis=1)
titulos_df_final


In [ ]:
#Podemos ver cuántos títulos con cada tipo de emoción clasificamos
titulos_df_final['emotion'].value_counts()

In [ ]:
#Y los podemos guardar como excel
titulos_df_final.to_excel('titulos.xlsx', index=False)

#### Un ejemplo en inglés usando TextBlob

Para que les quede de referencia por si quieren hacer algo en inglés en el TP.

In [ ]:
from textblob import TextBlob

In [ ]:
#Ejemplo de texto en inglés
texto_ej_1 = "I love learning about Big data"
texto_ej_2 = "I hate learning about Big data"

#Creamos un objeto TextBlob con el texto
blob1 = TextBlob(texto_ej_1)

#Obtenemos la polaridad del sentimiento (-1 a 1)
polarity1 = blob1.sentiment.polarity

#Clasificamos el sentimiento como positivo, negativo o neutro
def clasif_polarity(polarity):
    if polarity > 0:
        sentiment = "Positivo"
    elif polarity < 0:
        sentiment = "Negativo"
    else:
        sentiment = "Neutro"
    return sentiment

#Resultados
print(f"Texto: {texto_ej_1}")
print(f"Polaridad del sentimiento: {polarity1}")
print(f"Sentimiento: {clasif_polarity(polarity1)}")